# 05 · Heterogeneous Ensemble

Part of the study *Multi-Label 12-Lead ECG Classification on the PTB-XL Dataset* ([arXiv:2609.12803](https://arxiv.org/abs/2609.12803)). Run order, folder layout, and results: [project README](../README.md).

> Runtime: CPU (no training).

Five members are required: ResNet18-1D, xLSTM, Bidirectional Mamba, CWT-ViT-KAN, and ECGFounder (native 500 Hz from notebook 04).

Methods:
A. Equal soft voting (main baseline)
B. Class-specific val AUPRC weights (val fold 9 only)
C. Per-label LogisticRegression stacking (fit on fold 9)

Thresholds, weights, and the stacker are fit on **val fold 9**. Reported metrics, diversity, and 2×2 confusion matrices use **test fold 10**. Members must have written `results/<model>/seed_<seed>/test_predictions.csv` from the 01–04 Performance analysis cells.


In [ ]:
!pip install -q openpyxl scikit-learn pandas numpy matplotlib seaborn


In [ ]:
from google.colab import drive
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive') / 'ECG AI'
sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))

from ecg_common import (
    CLASSES,
    ENSEMBLE_MEMBERS,
    EVALUATE_TEST as DEFAULT_EVALUATE_TEST,
    apply_thresholds,
    align_prediction_frames,
    build_prediction_frame,
    class_auprc_weights,
    compute_multilabel_metrics,
    disagreement_rate,
    display_model_name,
    dump_json,
    ensure_dir,
    equal_soft_vote,
    error_overlap,
    fit_stacker,
    load_predictions,
    optimize_thresholds_f1,
    pairwise_prediction_correlation,
    per_class_metrics_table,
    plot_per_class_confusion_2x2,
    predict_stacker,
    save_prediction_csv,
    seed_results_dir,
    summary_table_row,
    threshold_vector,
    weighted_soft_vote,
)

EVALUATE_TEST = True
REPORT_SPLIT = 'test'
ACTIVE_SEED = 42
RESULTS_DIR = PROJECT_DIR / 'results'
OUT_DIR = ensure_dir(PROJECT_DIR / 'experiments' / 'ensemble' / f'seed_{ACTIVE_SEED}')
print('report', REPORT_SPLIT, '| EVALUATE_TEST', EVALUATE_TEST, '| seed', ACTIVE_SEED)
print('members', list(ENSEMBLE_MEMBERS))
print('fit on val fold 9; report on test fold 10')


In [ ]:
def load_member(split: str):
    frames = {}
    missing = []
    for key in ENSEMBLE_MEMBERS:
        path = seed_results_dir(RESULTS_DIR, key, ACTIVE_SEED) / f'{split}_predictions.csv'
        if not path.exists():
            missing.append(str(path))
            continue
        frames[key] = pd.read_csv(path)
    if missing:
        if split == 'test':
            hint = (
                'Missing test_predictions.csv. The seed may match, but fold 10 has not been scored yet. '
                'run the Performance analysis cell in notebooks 01-04 with EVALUATE_TEST=True '
                '(writes test from best.pt; do not retrain).\n'
            )
        else:
            hint = (
                f'Missing {split}_predictions.csv. All five models must be trained with the same seed.\n'
            )
        raise FileNotFoundError(hint + '\n'.join(missing))
    return frames

val_frames = load_member('val')
merged_val, val_probs, y_val = align_prediction_frames(val_frames, CLASSES)
print('val overlapping N', len(merged_val))

test_frames = load_member('test')
merged_test, test_probs, y_test = align_prediction_frames(test_frames, CLASSES)
print('test overlapping N', len(merged_test))
for k in ENSEMBLE_MEMBERS:
    print(k, 'val', val_frames[k].shape, 'test', test_frames[k].shape)


## Ensemble methods

Equal vote, class-AUPRC weighted vote, and stacking. Fit **only on val fold 9**; the frozen rule is applied to test fold 10.


In [ ]:
# A. Equal soft voting
p_equal_val = equal_soft_vote(val_probs, ENSEMBLE_MEMBERS)

# B. Class-specific validation AUPRC weights (fold 9 only)
w_class = class_auprc_weights(val_probs, y_val, ENSEMBLE_MEMBERS, CLASSES)
p_weighted_val = weighted_soft_vote(val_probs, w_class, ENSEMBLE_MEMBERS)
display(Markdown('#### Class-specific weights (rows=models, cols=classes)'))
display(pd.DataFrame(w_class, index=list(ENSEMBLE_MEMBERS), columns=list(CLASSES)))

# C. Per-label LogisticRegression stacking, fit on fold 9 only
stackers = fit_stacker(val_probs, y_val, ENSEMBLE_MEMBERS, CLASSES)
p_stack_val = predict_stacker(stackers, val_probs, ENSEMBLE_MEMBERS, CLASSES)
print('stacker fitted on validation fold 9; reported scores use test fold 10.')

ensemble_val = {
    'equal_soft_voting': p_equal_val,
    'weighted_soft_voting': p_weighted_val,
    'stacking': p_stack_val,
}
ens_thresholds = {
    name: optimize_thresholds_f1(y_val, prob, CLASSES) for name, prob in ensemble_val.items()
}
ensemble_test = {
    'equal_soft_voting': equal_soft_vote(test_probs, ENSEMBLE_MEMBERS),
    'weighted_soft_voting': weighted_soft_vote(test_probs, w_class, ENSEMBLE_MEMBERS),
    'stacking': predict_stacker(stackers, test_probs, ENSEMBLE_MEMBERS, CLASSES),
}
dump_json(
    {'class_auprc_weights': w_class.tolist(), 'thresholds': ens_thresholds, 'members': list(ENSEMBLE_MEMBERS)},
    OUT_DIR / 'ensemble_config.json',
)


## Metrics and diversity

Thresholds and weights come from fold 9; tables, diversity, and saved files use **test fold 10**.


In [ ]:
rows = []
pc_rows = []


def add_model(label, y_true, y_prob, thresholds, split):
    thr = threshold_vector(thresholds, CLASSES)
    metrics = compute_multilabel_metrics(y_true, y_prob, CLASSES, thresholds=thr)
    row = summary_table_row(label, metrics)
    row['split'] = split
    rows.append(row)
    t = per_class_metrics_table(metrics)
    t.insert(0, 'Model', label)
    t.insert(1, 'split', split)
    pc_rows.append(t)


rows = []
pc_rows = []


def add_model(label, y_true, y_prob, thresholds, split):
    thr = threshold_vector(thresholds, CLASSES)
    metrics = compute_multilabel_metrics(y_true, y_prob, CLASSES, thresholds=thr)
    row = summary_table_row(label, metrics)
    row['split'] = split
    rows.append(row)
    t = per_class_metrics_table(metrics)
    t.insert(0, 'Model', label)
    t.insert(1, 'split', split)
    pc_rows.append(t)


ind_thr = {}
for key in ENSEMBLE_MEMBERS:
    ind_thr[key] = optimize_thresholds_f1(y_val, val_probs[key], CLASSES)

for name, prob in ensemble_val.items():
    save_prediction_csv(
        build_prediction_frame(
            merged_val['ecg_id'], merged_val.get('patient_id', -1), y_val, prob, CLASSES,
            thresholds=ens_thresholds[name],
        ),
        OUT_DIR / 'predictions' / f'val_{name}_predictions.csv',
    )

for key in ENSEMBLE_MEMBERS:
    add_model(display_model_name(key), y_test, test_probs[key], ind_thr[key], 'test')
for name, prob in ensemble_test.items():
    label = 'Stacking Ensemble' if name == 'stacking' else display_model_name(name)
    add_model(label, y_test, prob, ens_thresholds[name], 'test')
    save_prediction_csv(
        build_prediction_frame(
            merged_test['ecg_id'], merged_test.get('patient_id', -1), y_test, prob, CLASSES,
            thresholds=ens_thresholds[name],
        ),
        OUT_DIR / 'predictions' / f'test_{name}_predictions.csv',
    )

pred_bin = {k: apply_thresholds(test_probs[k], threshold_vector(ind_thr[k], CLASSES)) for k in ENSEMBLE_MEMBERS}
corr = pairwise_prediction_correlation(test_probs, ENSEMBLE_MEMBERS)
disc = disagreement_rate(pred_bin, ENSEMBLE_MEMBERS)
ov = error_overlap(y_test, pred_bin, ENSEMBLE_MEMBERS)
display(Markdown('#### Pairwise probability correlation (test)'))
display(corr)
display(Markdown('#### Disagreement rate (test)'))
display(disc)
display(Markdown('#### Error overlap Jaccard (test)'))
display(ov)
corr.to_csv(OUT_DIR / 'diversity_correlation_test.csv')
disc.to_csv(OUT_DIR / 'diversity_disagreement_test.csv')
ov.to_csv(OUT_DIR / 'diversity_error_overlap_test.csv')

board = pd.DataFrame(rows)
pc = pd.concat(pc_rows, ignore_index=True)
display(Markdown('#### Comparison (test fold 10; thresholds/weights from val fold 9)'))
display(board)
board.to_csv(OUT_DIR / 'comparison.csv', index=False)
pc.to_csv(OUT_DIR / 'per_class_metrics.csv', index=False)
print('wrote', OUT_DIR)


## Confusion matrix (test, 2×2)


In [ ]:
ensure_dir(OUT_DIR / 'plots')
for name, prob in ensemble_test.items():
    pred = apply_thresholds(prob, threshold_vector(ens_thresholds[name], CLASSES))
    fig = plot_per_class_confusion_2x2(y_test, pred, CLASSES, title=f'Test 2x2 | {name}')
    fig.savefig(OUT_DIR / 'plots' / f'test_cm_2x2_{name}.png', dpi=150, bbox_inches='tight')
    plt.show()
